# Lesson 7 - Creating an A2A Sequential Chain Agent with ADK

Now that you have two active agents (Policy Agent and Research Agent), you will orchestrate them. In this lesson, you will use Google ADK's `SequentialAgent` to create a workflow where a user's query is processed by the Research Agent first, and then the Policy Agent. You will use `RemoteA2aAgent` (which acts as A2A Client) to connect to the servers you started in previous lessons.

![Sequential Workflow](sequential.png)

## 7.1. Start the A2A Servers

First, ensure that your `PolicyAgent` server is running.
- Open Terminal 1 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_policy_agent.py` (you don't need to go back to the previous lesson).

In [1]:
import os

from IPython.display import IFrame

url = os.environ.get("DLAI_LOCAL_URL").format(port=8888)
# Terminal 1: uv run a2a_policy_agent.py
IFrame(f"{url}terminals/1", width=800, height=200)

Second, ensure that your Research Agent is running.
- Open Terminal 2 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_research_agent.py` (you don't need to go back to the previous lesson).

In [2]:
# Terminal 2: uv run a2a_research_agent.py
IFrame(f"{url}terminals/2", width=800, height=200)

## 7.2. Define the Sequential Workflow

Here you will:
1.  Define two `RemoteA2aAgent` instances. These act as A2A clients that know how to communicate with your running servers via A2A.
2.  Create a `SequentialAgent` named `root_agent`. This agent contains the logic to route the workflow through the sub-agents.
3.  Use an `InMemoryRunner` to execute the flow with a specific prompt.


In [3]:
import os

from IPython.display import Markdown, display
from dotenv import load_dotenv
from google.adk.agents import SequentialAgent
from google.adk.agents.remote_a2a_agent import (
    RemoteA2aAgent,
)
from google.adk.runners import InMemoryRunner

import logging
import warnings

logging.disable(level=logging.WARNING)
warnings.filterwarnings("ignore")

In [4]:
load_dotenv()
host = os.environ.get("AGENT_HOST")
policy_port = os.environ.get("POLICY_AGENT_PORT")
research_port = os.environ.get("RESEARCH_AGENT_PORT")

In [5]:
policy_agent = RemoteA2aAgent(
    name="policy_agent",
    agent_card=f"http://{host}:{policy_port}",
)
print("\tℹ️", f"{policy_agent.name} initialized")

	ℹ️ policy_agent initialized


In [6]:
health_research_agent = RemoteA2aAgent(
    name="health_research_agent",
    agent_card=f"http://{host}:{research_port}",
)
print("\tℹ️", f"{health_research_agent.name} initialized")

	ℹ️ health_research_agent initialized


In [7]:
root_agent = SequentialAgent(
    name="root_agent",
    description="Healthcare Routing Agent",
    sub_agents=[
        health_research_agent,
        policy_agent,
    ],
)
print("\tℹ️", f"{root_agent.name} initialized")

	ℹ️ root_agent initialized


## 7.3. Run the Sequential Chain

The `InMemoryRunner` executes the agent. The prompt will trigger the Research Agent to find general info, and then (sequentially) the Policy Agent to check coverage details.

In [8]:
prompt = "How can I get mental health therapy?"

**Note:** It takes a few seconds for the output to display.

In [9]:
print("Running Healthcare Workflow Agent")

runner = InMemoryRunner(root_agent)

for event in await runner.run_debug(prompt, quiet=True):
    if event.is_final_response() and event.content:
        display(Markdown(event.content.parts[0].text))

Running Healthcare Workflow Agent


Taking the step to get mental health therapy is a highly beneficial and courageous decision. Because mental health needs vary from person to person, there is no one-size-fits-all approach to therapy. 

Here is a comprehensive, step-by-step guide on how to understand your options, find a therapist, and start treatment based on the latest available resources.

### 1. Understand Your Needs and Therapy Options
Before looking for a provider, it can be helpful to understand the different levels of care and types of therapy available so you can pinpoint what might work best for you.

**Levels of Care:**
*   **Outpatient Treatment:** The most common form of therapy, where you visit a clinic or log on virtually for weekly sessions while maintaining your daily life. This is ideal for mild to moderate symptoms and includes individual therapy, group therapy, or family counseling.
*   **Intensive Outpatient / Partial Hospitalization:** More structured programs where you visit a facility for several hours a day, multiple days a week.
*   **Residential or Inpatient Treatment:** Round-the-clock care in a residential facility for those who require continuous supervision and suffer from severe or long-term symptoms.
*   **Psychiatric Hospitalization:** For emergencies requiring stabilization, close monitoring, and immediate medical care.

**Common Types of Therapy (Modalities):**
*   **Cognitive Behavioral Therapy (CBT):** Highly goal-oriented, this focuses on identifying and changing negative, harmful thought patterns and behaviors. It is frequently used for depression, anxiety, OCD, and PTSD.
*   **Dialectical Behavior Therapy (DBT):** Combines CBT with mindfulness to help manage intense emotions, improve relationships, and develop emotional regulation. It is especially helpful for borderline personality disorder and self-harm behaviors.
*   **Acceptance and Commitment Therapy (ACT):** Focuses on accepting difficult thoughts and feelings rather than fighting them, while using mindfulness to improve overall psychological flexibility.
*   **Psychodynamic Therapy:** Explores how past experiences and unconscious patterns influence your current behaviors and feelings.
*   **Trauma-Focused Therapies:** Approaches like EMDR (Eye Movement Desensitization and Reprocessing) and somatic therapies are designed to help you process and heal from severe trauma.
*   **Exposure Therapy:** A subset of CBT that involves gradual exposure to fears in a safe environment to help reduce phobias and PTSD symptoms.
*   **Interpersonal Therapy (IPT):** Focuses on strengthening communication skills and navigating relationship challenges.

### 2. Where to Look for a Therapist
Once you know what kind of help you might need, you can begin your search. There are several reliable avenues:
*   **Online Therapist Directories:** The **Psychology Today Therapist Directory** is widely used and allows you to filter therapists by location, specialty, accepted insurance, gender, and the type of therapy they offer. Other regional directories exist as well, such as the BACP directory in the UK or the Australian Psychological Society directory. 
*   **Your Health Insurance:** Use your insurance provider’s online search tool or call the customer service number on the back of your ID card to find an in-network therapist. Keep in mind that insurance directories are not always updated frequently, so you may need to call the therapist directly to confirm they still accept your plan.
*   **Primary Care Physician (GP):** Discussing your mental health with your primary doctor is a great first step; they can provide referrals to trusted local therapists. 
*   **Mental Health Organizations:** Organizations like the National Alliance on Mental Illness (NAMI) offer resources and guidance on finding local help.

### 3. Reach Out and Filter Your Options
As you browse directories, look for a few therapists who specialize in your specific challenges (e.g., depression, relationship issues, adult ADHD). Keep logistical preferences in mind, such as whether you want in-person, virtual (telehealth), or hybrid appointments. You can also filter for therapists who share your racial, ethnic, or cultural background, or who are LGBTQ+ friendly.

**Making Contact:**
*   Choose three or four therapists who look like a good fit.
*   Send them an email or leave a voicemail. You can use a simple script like: *"I am looking for help with [my mood, relationship stress, anxiety, etc.]. Do you have current availability?"*. 
*   Be sure to mention your insurance type and provide the times you are available for a callback.
*   **Be Persistent:** Many therapists are solo practitioners and answer inquiries after their workday. If someone doesn’t get back to you after two tries, move on to the next person on your list.

### 4. Conduct a "Phone Screen"
Before committing to full sessions, schedule a brief (15-20 minute) consultation call. This is common practice and helps you gauge if the relationship will be a good fit. Helpful questions to ask during this call include:
*   How would you describe your therapy style?
*   Have you dealt with concerns similar to mine, and how frequently?
*   How does the therapy process work with you?
*   How frequently would you recommend visits for my needs?

### 5. Prioritize the "Therapeutic Alliance"
Research indicates that the single most important factor in a successful therapy outcome is the "therapeutic alliance"—the rapport, trust, and connection you build with your therapist. 

Finding the right therapist can sometimes feel like a guessing game, so trust your gut instinct. It is entirely okay to try a few sessions with someone and decide it isn't the right fit. Quality therapy should provide a safe, judgment-free space where you receive personalized care, learn coping skills, and feel supported.

# Mental Health Therapy Coverage Under Your Plan

Based on your Anthem Blue Cross gHIP HDHP plan, here's what you need to know about mental health therapy coverage:

## Coverage Details

**Outpatient Mental Health Services:**
- **In-Network Provider:** 10% coinsurance (after deductible)
- **Out-of-Network Provider:** 30% coinsurance (after deductible)

**Inpatient Mental Health Services:**
- **In-Network Provider:** 10% coinsurance (after deductible)
- **Out-of-Network Provider:** 30% coinsurance (after deductible)

## How to Access Services

1. **Find an In-Network Provider:**
   - Visit **includedhealth.com/google** or call **(855) 431-5540**
   - You can search for mental health providers in your network to minimize out-of-pocket costs

2. **No Referral Required:**
   - You can see a mental health specialist without needing a referral from your primary care physician

3. **Plan Network Options:**
   - **NY residents** (excluding Suffolk County): NY Blue Access PPO network
   - **GA residents:** GA Blue Open Access POS network
   - **All other areas:** Blue Card PPO network

## Important Cost Information

Remember that your costs apply **after you meet your deductible**:
- Individual deductible: \\$1,700 (in-network)
- Family deductible: \\$3,400 (in-network)

Your out-of-pocket maximum is \\$2,600 for individuals/\\$5,200 for families (in-network), after which the plan covers 100% of covered services.

For more detailed information about coverage limitations and exceptions, refer to your complete benefits booklet at **go/benefitdocuments**.

## 7.4. Resources

- [Google ADK Sequential Agents](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/)
- [Equivalent notebook in the course repo](https://github.com/holtskinner/A2AWalkthrough/blob/main/5_ADKSequentialAgent.ipynb)

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em>.</p>
</div>
